In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transformSales_person(SalesPerson_df):

    windowSpec_rw = Window.partitionBy("BusinessEntityID").orderBy("BusinessEntityID")
    SalesPerson_df = SalesPerson_df.withColumn("TerritoryID",F.when(F.col("TerritoryID").isNull(), 0).otherwise(F.col("TerritoryID"))).withColumn("SalesQuota",F.when(F.col("SalesQuota").isNull(), 0).otherwise(F.col("SalesQuota"))).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    SalesPerson_df = SalesPerson_df.filter(F.col("rw") == 1).drop("rw")
    SalesPerson_df = SalesPerson_df.withColumn("processed_timestamp", F.current_timestamp())
    SalesPerson_df = SalesPerson_df.select( 
        F.col("BusinessEntityID").cast(IntegerType()).alias("BusinessEntityID"),  
        F.col("TerritoryID").cast(IntegerType()).alias("TerritoryID"),
        F.col("SalesQuota").cast(DecimalType(19,4)).alias("SalesQuota"),
        F.col("Bonus").cast(DecimalType(19,4)).alias("Bonus"),
        F.col("CommissionPct").cast(DecimalType(19,4)).alias("CommissionPct"),
        F.col("SalesYTD").cast(DecimalType(19,4)).alias("SalesYTD"),
        F.col("SalesLastYear").cast(DecimalType(38,2)).alias("SalesLastYear"),
        F.col("rowguid"),
        F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
        F.col("_rescued_data").alias("_rescued_data"),
         F.col("processed_timestamp").cast(DateType()).alias("processed_timestamp"))
                                 
    return SalesPerson_df




if __name__ == "__main__":

    SalesPerson_tbl = dbutils.widgets.get("SalesPerson")
    SalesPerson_df = df = spark.read.table(SalesPerson_tbl)
    SalesPerson_df_tgt = transformSales_person(SalesPerson_df)
    SalesPerson_slv_tbl = dbutils.widgets.get("SalesPerson_tgt")
    SalesPerson_df_tgt.write.mode("overwrite").format("delta").partitionBy("ModifiedDate").saveAsTable(SalesPerson_slv_tbl)